# Yancc vs SFINCS Comparison - Part 2: Radial Ambipolar Scan

This notebook performs a radial scan over 15 points, calculating the ambipolar radial electric field ($E_r$) at each point using kinetic profiles loaded from a SFINCS profiles file.

In [ ]:
import numpy as np
import logging
import desc
import yancc
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import GlobalMaxwellian
from yancc.yancctools import sfincstools, yancctools

# Configure logging to see scan progress
logging.basicConfig(level=logging.INFO, force=True)

### 1. Load and Scale Kinetic Profiles

We load the polynomial coefficients from the SFINCS `profiles` file and wrap them in functions that provide SI units ($m^{-3}$ and $eV$).

In [ ]:
target_file = '/u/npablant/analysis/w7x/171207006/stelltran/run07/sfincs/t2.2273/profiles'

# Generate the raw polynomial functions (dimensionless coefficients)
funcs = sfincstools.generate_profile_functions(target_file)

# Define scaled functions for yancc (SFINCS profiles are in 10^20 m^-3 and keV)
# Species 1: Electrons, Species 2: Hydrogen
density_e = lambda r: funcs['species_1_density'](r) * 1e20
temp_e    = lambda r: funcs['species_1_temperature'](r) * 1e3

density_H = lambda r: funcs['species_2_density'](r) * 1e20
temp_H    = lambda r: funcs['species_2_temperature'](r) * 1e3

# Define global species objects
global_species = [
    GlobalMaxwellian(yancc.species.Electron, temperature=temp_e, density=density_e),
    GlobalMaxwellian(yancc.species.Hydrogen, temperature=temp_H, density=density_H)
]

print("Profiles loaded and species defined.")

### 2. Setup Equilibrium and Grids

We use the standard W7-X equilibrium and define the resolution for the velocity space and radial scan.

In [ ]:
# Load W7-X equilibrium
eq = desc.examples.get("W7-X")

# Define radial grid (15 points)
rho_grid = np.linspace(0.2, 0.8, 15)

# Define velocity grids
speedgrid = MaxwellSpeedGrid(nx=5)
pitchgrid = UniformPitchAngleGrid(nxi=65)

### 3. Execute Radial Scan with Ambipolar Er Search

The `scan_ambipolar_profile` function will iterate through each radial point and find the $E_r$ roots where the net charge flux is zero.

In [ ]:
options = {
    "erho_min": -15000.0,  # Coarse scan range [V/m]
    "erho_max": 15000.0,
    "erho_num": 21,         # Points in coarse scan
    "nt": 17,               # Poloidal resolution
    "nz": 33,               # Toroidal resolution
    "rtol": 1e-4
}

results = yancctools.scan_ambipolar_profile(
    rho_grid,
    eq_type="desc",
    eq_data=eq,
    pitchgrid=pitchgrid,
    speedgrid=speedgrid,
    global_species=global_species,
    options=options
)

print(f"Radial scan completed. RunID: {results['runid']}")

### 4. Visualize Results

In [ ]:
yancctools.plot_ambipolar_summary(results, global_species)

In [ ]:
from yancc.yancctools import mirhdf5
mirhdf5.dictToHdf5(results, '/u/npablant/analysis/yancc/runs/R8-PIS-260313-225925/R8-PIS-260313-225925.hdf5')